## Practical work on the conversion of sampling rate and STFT


In [ ]:
import os, sys, wave, struct

import numpy as np
import pyaudio
import pandas as pd
import matplotlib.pyplot as plt

from copy import deepcopy
from math import ceil
from scipy.io.wavfile import write
import scipy
import time



### Section 1.1: Sampling Rate Conversion Parameters
%%(How to draw?)%%

To convert the audio signal from an initial sampling frequency of $f_{in} = 48$ kHz to a target frequency of $f_{out} = 32$ kHz, we must apply a rational sampling rate conversion. The conversion factor is given by the ratio of the two frequencies:
$$\frac{f_{out}}{f_{in}} = \frac{32000}{48000} = \frac{2}{3} = \frac{L}{M}$$

From this rational factor, we identify the two fundamental parameters for our system:
* **Interpolation factor ($L$):** $2$ (upsampling)
* **Decimation factor ($M$):** $3$ (downsampling)

To prevent spectral distortions during this process, a low-pass filter must be inserted between the upsampler and the downsampler. This filter operates at the intermediate sampling rate of $f_{int} = L \cdot f_{in} = 96$ kHz. 

The filter serves a dual purpose:
1. **Anti-imaging:** It removes the spectral images created by the zero-insertion (upsampling by $L$), requiring a normalized cutoff frequency $\nu \le \frac{1}{2L}$.
2. **Anti-aliasing:** It prevents frequency overlap during the subsequent downsampling by $M$, requiring a normalized cutoff frequency $\nu \le \frac{1}{2M}$.

The ideal normalized cutoff frequency $\nu_c$ is dictated by the most restrictive of these two conditions:
$$\nu_c = \min\left( \frac{1}{2L}, \frac{1}{2M} \right) = \min\left( \frac{1}{4}, \frac{1}{6} \right) = \frac{1}{6}$$

In the physical frequency domain, considering the intermediate sampling rate of 96 kHz, this corresponds to an absolute cutoff frequency of:
$$f_c = \nu_c \cdot f_{int} = \frac{1}{6} \cdot 96 \text{ kHz} = 16 \text{ kHz}$$

This mathematical result perfectly aligns with the Nyquist theorem: to sample a signal at 32 kHz without aliasing, we must strictly band-limit the signal to its Nyquist frequency, which is exactly $16$ kHz.

### Section 1.2: Synthetization of the H(n)

Synthesization of an impulse reponse h(n) appropriate for this conversion of sampling rate using the Remez method:


In [ ]:
F_start = 48  # initial sampling rate 
F_final = 32  # final sampling rate

L = 2
M = 3
alfa = 0.1     # defined by us, to find the transiction band

nu_real = min(1/(2*L), 1/(2*M))
print("nu real: ", nu_real)
nu_c = nu_real - nu_real*alfa
nu_a = nu_real + nu_real*alfa

even_length = 90   # defined by us

print(f"Interval of nu: {nu_c:.3F} - {nu_a}")


In [ ]:
h = scipy.signal.remez(even_length, [0,nu_c,nu_a,.5], [L, 0])

w, q = scipy.signal.freqz(h, worN=1024)
nu = w/(2*np.pi) # normalized frequency
H_dB = 20*np.log10(np.abs(q) + 1e-12) # magnitude in dB, adding a small value to avoid log(0)

#plotting the filter
plt.plot(nu, H_dB)
plt.title('Frequency response of the designed filter')
plt.xlabel('Normalized Frequency (xπ rad/sample)')
plt.ylabel('Magnitude (dB)')
plt.grid()
plt.show()

As it is shown in the plotted result, we obtain a difference of almost 50 dB between the pass-band and the stop-band. 


### Section 1.3: Implementation and Validation of the Resampling Chain

Here we implement the entire sample rate conversion chain on the real audio signal `caravan_48khz.wav`. The goal is to reduce the sample rate from 48 kHz to 32 kHz by sequentially applying the three key operations studied: upsampling by a factor of $L=2$ (bringing the signal to 96 kHz), low-pass anti-aliasing/anti-imaging filtering, and subsequent decimation by a factor of $M=3$, verifying the effectiveness of the system in both the time and frequency domains.

In [ ]:
# read the audio file and extract the signal
sample_rate, x = scipy.io.wavfile.read('caravan_48khz.wav')
print(f"Original sample rate: {sample_rate} Hz")
print(f"Original signal length: {len(x)} samples")


In [ ]:
start_time = time.time()
# 1. Upsampling by a factor of L
x_upsampled = np.zeros(len(x) * L) #96000 di 0
x_upsampled[::L] = x

# 2. Filtering the upsampled signal with the designed low-pass filter
y_upsampled = scipy.signal.lfilter(h, 1, x_upsampled)

# 3. Downsampling by a factor of M
y_downsampled = y_upsampled[::M]

end_time = time.time()
print(f"Time: {end_time - start_time} seconds")

F_final = sample_rate * L / M
print(f"Final sample rate after resampling: {F_final} Hz")

#plot of signals
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)    
plt.plot(x[:])  # plot the first 3000 samples of the original signal
plt.title('Original Signal')
plt.subplot(2, 1, 2)
plt.plot(y_downsampled[:])  # plot the first 2000 samples of the downsampled signal
plt.title('Downsampled Signal')
plt.tight_layout()
plt.show()

#plot spectrograms
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.specgram(x,NFFT=1024, Fs=sample_rate, noverlap=512)
plt.title('Spectrogram of Original Signal')
plt.ylim(0, sample_rate/2)  # limit y-axis to Nyquist frequency of the original signal
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')

plt.subplot(2, 1, 2)
plt.specgram(y_downsampled, NFFT=1024, Fs=F_final, noverlap=512)
plt.title('Spectrogram of Downsampled Signal')
plt.tight_layout()
plt.ylim(0, 24000)  # limit y-axis to Nyquist frequency of the downsampled signal
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')
plt.show()


Implementation and Results Analysis:

**Time-Domain Analysis**
The time-domain plots compare the original 48 kHz waveform against the downsampled 32 kHz signal. As visually confirmed, the macroscopic envelope, transient peaks, and overall temporal dynamics of the audio track are perfectly preserved. Furthermore, the amplitude range remains consistent, indicating that the filter's DC gain appropriately compensated for the energy dilution inherently caused by the zero-insertion step.

**Frequency-Domain Analysis (Spectrograms)**
The spectrogram plots provide a definitive validation of the multirate system's frequency response:
* The **Original Signal** spectrogram displays natural spectral components extending across the entire available bandwidth, up to its Nyquist frequency of $24$ kHz.
* The **Downsampled Signal** spectrogram reveals a sharp, clean cut-off at exactly $16$ kHz, which corresponds to the new Nyquist frequency ($32 \text{ kHz} / 2$). The anti-aliasing filter effectively suppressed all original spectral energy above $16$ kHz prior to decimation. As a direct result, the region above $16$ kHz is completely empty, proving that no high-frequency components folded back into the lower spectrum. This guarantees a high-fidelity downsampling completely free of aliasing artifacts.

## Optimal Implementation

### Section 1.4: The Noble Identity for Decimation



**1. Theoretical Equivalence**
The block diagram in Figure 1 illustrates a fundamental property in multirate digital signal processing known as the **Noble Identity for Decimation**. This mathematical identity states that filtering a signal with a transfer function $H(z^M)$ prior to downsampling by a factor of $M$ is strictly equivalent to first downsampling the signal by $M$ and subsequently filtering it with $H(z)$.

Applying this theorem to a pure delay system, we can establish the following equivalence:
* **System A:** Applying a delay of $M$ samples (represented by $z^{-M}$ in the z-domain) *before* the decimation block ($\downarrow M$).
* **System B:** Applying a delay of $1$ sample ($z^{-1}$) *after* the decimation block ($\downarrow M$).

Both configurations yield the exact same output signal. For our specific decimation factor of $M = 3$, applying a $z^{-3}$ delay prior to downsampling is perfectly equivalent to applying a $z^{-1}$ delay after downsampling.

**2. Computational Significance**
The engineering value of this equivalence lies in computational optimization. By applying the Noble Identity (transitioning from System A to System B), the mathematical operations (delays or filtering) are pushed to the right side of the decimation block. 

As a result, the operations are performed at a drastically reduced sampling rate ($F_{in}/M$ instead of $F_{in}$). This principle represents the theoretical foundation of **Polyphase Filter Synthesis**, allowing a complex, high-rate filter to be decomposed into multiple parallel sub-filters operating at a lower frequency.

### Sections 1.5 & 1.6: Polyphase Decomposition and Performance Benchmark

**1. Theoretical Framework: The Double Polyphase Decomposition**
The core objective of this section is to practically implement the Noble Identities to optimize the basic resampling algorithm, maximizing computational speed without altering the signal. The theoretical approach relies on **two successive polyphase decompositions**:

* **First Decomposition (Type II for Upsampling, $L=2$):** Instead of zero-padding the input signal and filtering, the prototype filter $h(n)$ is split into two distinct sub-filters: $h_0$ (containing the even-indexed coefficients) and $h_1$ (containing the odd-indexed coefficients).
* **Second Decomposition (Type I for Downsampling, $M=3$):** Instead of computing the output of the previous filters only to discard $M-1$ samples, $h_0$ and $h_1$ are further split into 3 parallel branches each.

The resulting architecture is a matrix of **$L \times M = 6$ sub-filters**. The immense advantage of this structure is that all 6 sub-filters operate at the lowest possible sampling rate in the chain ($16$ kHz). They compute *only* the mathematical operations required for the samples that will actually be retained in the final audio output.


In the implementation we use the `scipy.signal.upfirdn` function, which internally handles the polyphase matrix.
*Run the code cell below to execute both implementations and compare their performance:*

In [ ]:
# read the audio file and extract the signal
sample_rate, x = scipy.io.wavfile.read('caravan_48khz.wav')

#compare the time of the two methods with printing the time taken for each method
print(f"COMPARISON OF TIME TAKEN FOR MANUAL RESAMPLING AND POLYPHASE RESAMPLING:")

start_time = time.time()

#Manual resampling
x_upsampled = np.zeros(len(x) * L) 
x_upsampled[::L] = x
y_upsampled = scipy.signal.lfilter(h, 1, x_upsampled)
y_downsampled = y_upsampled[::M]

end_time = time.time()
print(f"Manual time resampling: {end_time - start_time} seconds")

#Polyphase resampling using scipy.signal.upfirdn
start_time = time.time()
y = scipy.signal.upfirdn(h, x, up=L, down=M)

end_time = time.time()
print(f"Polyphase time resampling: {end_time - start_time} seconds")

### Conclusion: Performance Comparison
The execution times clearly demonstrate the computational superiority of the polyphase implementation. By breaking the filter into a polyphase matrix, it performs calculations only on non-zero samples and directly at the lowest sampling rate. 
As a result, the polyphase method is significantly faster while guaranteeing the exact same mathematical and acoustic output.